Cell 1 — Imports + quick sanity prints

In [4]:
import os  # file checks
import time  # timing

import cv2  # webcam + drawing
import mediapipe as mp  # mediapipe main package

print("✅ Imported cv2 and mediapipe")  # track
print("OpenCV version:", cv2.__version__)  # track
print("mediapipe path:", mp.__file__)  # track
print("Has mp.solutions?:", hasattr(mp, "solutions"))  # track
print("Has mp.tasks?:", hasattr(mp, "tasks"))  # track

print("CWD:", os.getcwd())  # track
print("Shadow check - mediapipe.py exists here?:", os.path.exists("mediapipe.py"))  # track
print("Shadow check - mediapipe folder exists here?:", os.path.isdir("mediapipe"))  # track


✅ Imported cv2 and mediapipe
OpenCV version: 4.12.0
mediapipe path: C:\Users\HP\anaconda3\envs\DL\Lib\site-packages\mediapipe\__init__.py
Has mp.solutions?: False
Has mp.tasks?: True
CWD: C:\Users\HP\Desktop\Jupyter Notebooks\Primed\rPPG Model\Examples\Data Preprocessing
Shadow check - mediapipe.py exists here?: False
Shadow check - mediapipe folder exists here?: False


Cell 2 — Download the face model

In [5]:
import urllib.request  # used to download the model file

MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/face_detector/"
    "blaze_face_short_range/float16/latest/blaze_face_short_range.tflite"
)  # model download URL

MODEL_PATH = "blaze_face_short_range.tflite"  # local filename for the model

print("📦 Model path:", os.path.abspath(MODEL_PATH))  # show where model will be saved

if not os.path.exists(MODEL_PATH):  # check if model already exists
    print("⬇️ Downloading face detector model...")  # progress print
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)  # download model
    print("✅ Model downloaded")  # done print
else:
    print("✅ Model already exists (skip download)")  # done print


📦 Model path: C:\Users\HP\Desktop\Jupyter Notebooks\Primed\rPPG Model\Examples\Data Preprocessing\blaze_face_short_range.tflite
⬇️ Downloading face detector model...
✅ Model downloaded


Cell 3 — Create the MediaPipe Tasks FaceDetector

In [6]:
from mediapipe.tasks.python import vision  # MediaPipe vision tasks

print("✅ Imported mediapipe.tasks.python.vision")  # track

BaseOptions = mp.tasks.BaseOptions  # base options class
FaceDetectorOptions = vision.FaceDetectorOptions  # face detector options class
RunningMode = vision.RunningMode  # running mode enum

print("⚙️ Creating FaceDetector options...")  # track

options = FaceDetectorOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),  # use model file
    running_mode=RunningMode.VIDEO,  # video mode (needs timestamps)
    min_detection_confidence=0.5,  # ignore weak detections
    min_suppression_threshold=0.3  # overlap suppression threshold
)

print("✅ FaceDetector options created")  # track


✅ Imported mediapipe.tasks.python.vision
⚙️ Creating FaceDetector options...
✅ FaceDetector options created


Cell 4 — Function to run webcam + detect faces + draw boxes

In [14]:
def run_webcam_face_detection(camera_index=0, window_name="Koolac", print_every=30):
    print(f"🎥 Opening webcam index: {camera_index}")

    cap = cv2.VideoCapture(camera_index)

    if not cap.isOpened():
        print("❌ ERROR: Could not open webcam. Try camera_index=1 or 2.")
        return

    # Create a visible window (helps on some setups)
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.moveWindow(window_name, 50, 50)

    print("✅ Webcam opened")
    print("➡️ Press 'q' or ESC to quit, or close the window (X).")

    frame_count = 0
    t0 = time.perf_counter()

    try:
        with vision.FaceDetector.create_from_options(options) as detector:
            print("✅ FaceDetector initialized")

            while True:
                ok, frame_bgr = cap.read()
                if not ok:
                    print("⚠️ WARNING: Failed to read frame")
                    continue

                frame_count += 1

                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
                timestamp_ms = int((time.perf_counter() - t0) * 1000)

                result = detector.detect_for_video(mp_image, timestamp_ms)

                if result.detections:
                    for det in result.detections:
                        bbox = det.bounding_box
                        x, y, w, h = int(bbox.origin_x), int(bbox.origin_y), int(bbox.width), int(bbox.height)

                        cv2.rectangle(frame_bgr, (x, y), (x + w, y + h), (0, 255, 0), 2)

                        score = det.categories[0].score if det.categories else 0.0
                        cv2.putText(frame_bgr, f"{score:.2f}", (x, max(0, y - 10)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

                    if frame_count % print_every == 0:
                        print(f"🙂 Faces: {len(result.detections)} | frame={frame_count} | ts={timestamp_ms}ms")

                cv2.imshow(window_name, frame_bgr)

                # If user closes the OpenCV window, stop
                if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
                    print("🛑 Window closed (X). Exiting...")
                    break

                key = cv2.waitKey(5) & 0xFF
                if key == ord('q') or key == 27:  # 27 = ESC
                    print("🛑 Quit key pressed (q or ESC). Exiting...")
                    break

    except KeyboardInterrupt:
        print("🛑 Interrupted from Jupyter (Kernel → Interrupt). Exiting...")

    finally:
        cap.release()
        cv2.destroyAllWindows()
        print("✅ Webcam released and windows closed")


In [15]:
run_webcam_face_detection(camera_index=0, window_name="Koolac", print_every=30)


🎥 Opening webcam index: 0
✅ Webcam opened
➡️ Press 'q' or ESC to quit, or close the window (X).
✅ FaceDetector initialized
🙂 Faces: 1 | frame=30 | ts=1015ms
🙂 Faces: 1 | frame=60 | ts=1999ms
🙂 Faces: 1 | frame=90 | ts=3018ms
🙂 Faces: 1 | frame=120 | ts=3997ms
🙂 Faces: 1 | frame=150 | ts=4996ms
🙂 Faces: 1 | frame=180 | ts=6010ms
🙂 Faces: 1 | frame=210 | ts=7007ms
🙂 Faces: 1 | frame=240 | ts=8001ms
🙂 Faces: 1 | frame=270 | ts=9014ms
🙂 Faces: 1 | frame=300 | ts=10011ms
🙂 Faces: 1 | frame=330 | ts=11010ms
🙂 Faces: 1 | frame=360 | ts=11995ms
🙂 Faces: 1 | frame=390 | ts=13009ms
🙂 Faces: 1 | frame=420 | ts=14004ms
🙂 Faces: 1 | frame=450 | ts=15019ms
🙂 Faces: 1 | frame=480 | ts=16003ms
🙂 Faces: 1 | frame=510 | ts=17022ms
🙂 Faces: 1 | frame=540 | ts=18004ms
🙂 Faces: 1 | frame=570 | ts=19007ms
🙂 Faces: 1 | frame=600 | ts=20003ms
🙂 Faces: 1 | frame=630 | ts=21015ms
🙂 Faces: 1 | frame=660 | ts=22013ms
🙂 Faces: 1 | frame=690 | ts=23022ms
🙂 Faces: 1 | frame=720 | ts=24002ms
🙂 Faces: 1 | frame=750 | 